# 求和知识库

> 大作业二：求和知识库  
> 姓名：_边琪雯_  学号：_3210100920_  日期：_2026.5.9_

## 项目简介
收集了多种不同的求和方法，涵盖迭代、函数式、递归、数值稳定、向量化、特殊类等范式。  
每种方法包含：名称、原理、适用场景、易错点、AI使用标注。

## 如何使用本笔记本
- 左侧目录可点击跳转
- 每个方法独立一个代码单元格，可直接运行测试
- 最后有统一测试单元格，验证边界情况

## 目录

点击下方链接跳转到对应方法：

1. [方法1：内置 sum 函数](#method1)
2. [方法2：for 循环累加](#method2)
3. [方法3：生成器表达式 + sum](#method3)
4. [方法4：reduce + lambda](#method4)
5. [方法5：直接递归（分治）](#method5)
6. [方法6：math.fsum（高精度）](#method6)
7. [方法7：Kahan 求和算法](#method7)
8. [方法8：numpy.sum（向量化）](#method8)
9. [方法9：pandas.Series.sum](#method9)
10. [方法10：等差数列公式（O(1)）](#method10)
11. [方法11：矩阵点积（numpy.dot）](#method11)

### 其他跳转
- [统一测试区块](#test-block)
- [抽象与归纳](#summary-block)

<a id="method1"></a>
### 方法1：内置 sum 函数

- **原理**：python 自带的 sum 函数，速度最快  
- **适用场景**：适用于大部分可迭代对象（list、tuple、range等），数据量在百万级以内时性能很好
- **易错点**：浮点数累加会有累积舍入误差；空可迭代对象返回 0 ；对非数值类型（如字符串）会报错
- **AI协助**：0%

In [17]:
def f_sum_1(data):
    return sum(data)

if __name__ == "__main__":
    print(f_sum_1([1,2,3,4]))

10


<a id="method2"></a>
### 方法2：for 循环累加

- **原理**：显式循环，使用 for 语句遍历每个元素，累加到变量中
- **适用场景**：适合任何可迭代对象，适用于更灵活的控制（中间加判断、打印等）  
- **易错点**：浮点数有舍入误差；需要手动初始化累加变量为 0；空列表时返回初始值 0  
- **AI协助**：0%

In [18]:
def f_sum_2(data):
    total = 0
    for x in data:
        total += x
    return total

if __name__ == "__main__":
    print(f_sum_2([1,2,3,4]))

10


<a id="method3"></a>
### 方法3：生成器表达式 + sum

- **原理**：生成器表达式逐个产生元素，sum 函数进行累加，不创建中间列表  
- **适用场景**：处理大数据流时节省内存，例如处理文件行、range 大范围数值  
- **易错点**：生成器只能遍历一次；表达式写法需要注意括号；与普通 sum(list) 相比速度略慢（生成器开销）  
- **AI协助**：10% 易错点解释

In [19]:
def f_sum_3(data):
    return sum(x for x in data)

if __name__ == "__main__":
    print(f_sum_3([1,2,3,4]))

10


<a id="method4"></a>
### 方法4：reduce + lambda

- **原理**：使用 functools.reduce 反复将二元操作（lambda a,b: a+b）应用于序列，归约为单一值  
- **适用场景**：函数式编程风格；求和只是归约的一个特例，学习 reduce 用法
- **易错点**：需要导入 functools；空序列必须提供初始值（如 0），否则报错；lambda 比内置 sum 慢  
- **AI协助**：90%

In [20]:
from functools import reduce

def f_sum_4(data):
    return reduce(lambda a, b: a + b, data, 0)

if __name__ == "__main__":
    print(f_sum_4([1,2,3,4]))

10


<a id="method5"></a>
### 方法5：直接递归（分治）

- **原理**：递归地将问题分解为“第一个元素 + 剩余元素的和”，直到序列为空  
- **适用场景**：教学演示递归思想时使用；小规模数据（通常长度 < 1000）  
- **易错点**：Python 默认递归深度约 1000，对大列表会引发 RecursionError；每次递归复制列表切片效率极低  
- **AI协助**：20% 方法、易错点

In [21]:
def f_sum_5(data):
    # 请实现递归求和（小规模演示）
    if not data:
        return 0
    return data[0] + f_sum(data[1:])

if __name__ == "__main__":
    print(f_sum_5([1,2,3,4]))

10.0


<a id="method6"></a>
### 方法6：math.fsum（高精度）

- **原理**：使用 Shewchuk 算法，通过部分和补偿减少浮点舍入误差，返回高精度浮点结果  
- **适用场景**：对浮点数求和时要求高精度（如金融计算、科学计算）  
- **易错点**：速度比普通 sum 慢（约 2-5 倍）；只接受浮点数列表；整数也会被转为 float 返回  
- **AI协助**：100%

In [22]:
import math

def f_sum_6(data):
    return math.fsum(data)

if __name__ == "__main__":
    print(f_sum_6([0.1, 0.2, 0.3]))

0.6


<a id="method7"></a>
### 方法7：Kahan 求和算法

- **原理**：通过一个补偿变量记录小数部分的丢失，逐次修正求和结果，能有效减少浮点累加误差  
- **适用场景**：对大量浮点数求和且不希望牺牲过多速度时（比 math.fsum 快一些）  
- **易错点**：实现需注意变量顺序和类型；对极端条件（如巨大数值差）仍可能有微小误差；不适合整数求和  
- **AI协助**：100%

In [23]:
def f_sum_7(data):
    s = 0.0
    c = 0.0
    for x in data:
        y = x - c
        t = s + y
        c = (t - s) - y
        s = t
    return s

if __name__ == "__main__":
    print(f_sum_7([0.1, 0.2, 0.3]))

0.6


<a id="method8"></a>
### 方法8：numpy.sum（向量化）

- **原理**：调用 NumPy 的 C 语言实现的向量化累加，利用 SIMD 指令加速  
- **适用场景**：处理大型数组（百万级及以上），或已经使用 NumPy 数组进行科学计算时  
- **易错点**：需要安装 numpy；输入普通 list 时会先转换为 ndarray，有小量开销；对空数组返回 0.0 或 0 取决于 dtype  
- **AI协助**：30% 提供思路和介绍，本身用过

In [24]:
import numpy as np

def f_sum_8(data):
    return np.sum(data)

if __name__ == "__main__":
    print(f_sum_8([1,2,3,4]))

10


<a id="method9"></a>
### 方法9：pandas.Series.sum

- **原理**：将数据转换为 pandas Series，然后调用其 sum 方法，同样使用向量化操作  
- **适用场景**：数据已经在 pandas 的 DataFrame 或 Series 中，进行数据清洗或分析时  
- **易错点**：需要安装 pandas；对普通 list 转换有额外开销；默认跳过缺失值（NaN），可能掩盖数据问题  
- **AI协助**：30% 提供思路和介绍，本身用过

In [25]:
import pandas as pd

def f_sum_9(data):
    return pd.Series(data).sum()

if __name__ == "__main__":
    print(f_sum_9([1,2,3,4]))

10


<a id="method10"></a>
### 方法10：等差数列公式（O(1)）

- **原理**：利用等差数列求和公式 `n*(首项+末项)/2` 直接计算，不遍历元素  
- **适用场景**：数据恰好是等差数列，且已知首项、末项和项数；常用于 range 生成的数据  
- **易错点**：不通用，非等差数据会得出错误结果；代码中需要增加验证或明确说明使用前提；整数除法需注意类型  
- **AI协助**：100%

In [26]:
def f_sum_10(data):
    if not data:
        return 0
    first = data[0]
    last = data[-1]
    n = len(data)
    return n * (first + last) // 2 if all(isinstance(x, int) for x in data) else n * (first + last) / 2

if __name__ == "__main__":
    print(f_sum_10([1,2,3,4,5]))

15


<a id="method11"></a>
### 方法11：矩阵点积（numpy.dot）

- **原理**：构造一个全 1 的向量，与原数据向量做点积，等价于求和  
- **适用场景**：在矩阵运算环境下使用  
- **易错点**：需要 numpy；构造全 1 向量需要额外内存；本质还是向量化累加，没有性能优势  
- **AI协助**：100%

In [27]:
import numpy as np

def f_sum_11(data):
    ones = np.ones(len(data))
    return np.dot(ones, data)

if __name__ == "__main__":
    print(f_sum_11([1,2,3,4]))

10.0


<a id="test-block"></a>
## 统一测试与边界验证

In [28]:
test_cases = {
    "空列表": [],
    "单元素": [42],
    "负数": [-5, -10, -15],
    "浮点数": [0.1, 0.2, 0.3],
    "混合": [10, -3, 2.5, 0]
}

sum_functions = [
    f_sum_1,
    f_sum_2,
    f_sum_3,
    f_sum_4,
    f_sum_5,
    f_sum_6,
    f_sum_7,
    f_sum_8,
    f_sum_9,
    f_sum_10,
    f_sum_11
]

for func in sum_functions:
    print(f"\n测试 {func.__name__}:")
    for name, data in test_cases.items():
        try:
            result = func(data)
            print(f"  {name}: {result}")
        except Exception as e:
            print(f"  {name}: 错误 -> {e}")


测试 f_sum_1:
  空列表: 0
  单元素: 42
  负数: -30
  浮点数: 0.6
  混合: 9.5

测试 f_sum_2:
  空列表: 0
  单元素: 42
  负数: -30
  浮点数: 0.6000000000000001
  混合: 9.5

测试 f_sum_3:
  空列表: 0
  单元素: 42
  负数: -30
  浮点数: 0.6
  混合: 9.5

测试 f_sum_4:
  空列表: 0
  单元素: 42
  负数: -30
  浮点数: 0.6000000000000001
  混合: 9.5

测试 f_sum_5:
  空列表: 0
  单元素: 42.0
  负数: -30.0
  浮点数: 0.6
  混合: 9.5

测试 f_sum_6:
  空列表: 0.0
  单元素: 42.0
  负数: -30.0
  浮点数: 0.6
  混合: 9.5

测试 f_sum_7:
  空列表: 0.0
  单元素: 42.0
  负数: -30.0
  浮点数: 0.6
  混合: 9.5

测试 f_sum_8:
  空列表: 0.0
  单元素: 42
  负数: -30
  浮点数: 0.6000000000000001
  混合: 9.5

测试 f_sum_9:
  空列表: 0
  单元素: 42
  负数: -30
  浮点数: 0.6000000000000001
  混合: 9.5

测试 f_sum_10:
  空列表: 0
  单元素: 42
  负数: -30
  浮点数: 0.6000000000000001
  混合: 20.0

测试 f_sum_11:
  空列表: 0.0
  单元素: 42.0
  负数: -30.0
  浮点数: 0.6000000000000001
  混合: 9.5


<a id="summary-block"></a>
## 抽象与归纳

六类范式：
- 迭代类（for循环、生成器表达式）
- 函数式（sum、reduce）
- 递归类（直接递归）
- 数值稳定（fsum、Kahan）
- 向量化（numpy、pandas、点积）
- 特殊类（等差数列公式）

两类核心思路：
- 按顺序求和。for 循环、reduce、递归、numpy.sum 等，核心是把数据里的数一个接一个往总数上加；
- 使用数学公式。只有等差数列公式（方法10）属于这类，不遍历数据，直接用首项、末项、项数套公式出结果；

## 完成清单（对照量表）

- [☑️] 第一部分：判定标准已写在笔记本开头
- [☑️] 第二部分：已覆盖11种方法，6个范式
- [☑️] 第三部分：每个方法都有五项元数据
- [☑️] 第四部分：已写抽象归纳
- [☑️] 第五部分：目录可跳转，结构清晰
- [☑️] 第六部分：元认知反思
- [☑️] 第七部分：同伴互评